# Environment Setup

In [1]:
# REMEMBER RUN THIS CODE AND IT WILL RESTART THE KERNEL
# RUN THIS CELL AGAIN AFTER THE RESTART AND IT WILL SAY "✅ Environment is Ready!"

import os
import sys

# 1. Define the "Check File"
DONE_FLAG = "/content/env_fixed.flag"

if not os.path.exists(DONE_FLAG):
    print("🔧 Fixing Environment & Installing Model Libraries...")

    # 2. Install uv (Instant)
    !pip install -q uv

    !uv pip install --system --quiet \
        "numpy<2.0.0" \
        "pandas<2.2.0" \
        "pyarrow>=14.0.0" \
        transformers \
        timm \
        faiss-gpu-cu12 \
        duckdb \
        mlflow \
        dagshub \
        psutil \
        webdataset

    # 4. Mark as done
    !touch {DONE_FLAG}

    # 5. AUTOMATIC RESTART
    print("🔄 Restarting Kernel to load new libraries... (Please wait a moment)")
    os.kill(os.getpid(), 9)

else:
    # After the restart, this block runs.
    import numpy as np
    import pandas as pd
    print("✅ Environment is Ready!")

✅ Environment is Ready!


In [15]:
# ==========================================
# 1. STANDARD LIBRARY & SYSTEM
# ==========================================
import os
import sys
import gc
import time
import shutil
import sqlite3
import psutil
import resource
import gzip
import csv
import glob
import functools
import subprocess
from collections import Counter
from pathlib import Path
from typing import List, Dict, Tuple, Optional, Any
from io import BytesIO

# ==========================================
# 2. DATA SCIENCE & METRICS
# ==========================================
import numpy as np
import pandas as pd
import duckdb
import pyarrow as pa
import pyarrow.parquet as pq

# ==========================================
# 3. PYTORCH & DEEP LEARNING
# ==========================================
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
import webdataset as wds  # Critical for streaming shards

# ==========================================
# 4. HUGGING FACE & MODEL ARCHITECTURES
# ==========================================
import transformers
from transformers import (
    AutoImageProcessor,
    AutoModel,
    CLIPModel,
    CLIPProcessor,
    SiglipModel,
    SiglipProcessor
)
from huggingface_hub import list_repo_files, hf_hub_url

# ==========================================
# 5. IMAGE PROCESSING
# ==========================================
from PIL import Image

# ==========================================
# 6. VECTOR SEARCH & TRACKING
# ==========================================
import faiss
import mlflow
import dagshub

# ==========================================
# 7. UTILITIES
# ==========================================
from tqdm.auto import tqdm
import urllib

# ==========================================
# 8. COLAB SETUP
# ==========================================
try:
    from google.colab import drive, userdata
except ImportError:
    print("Google Colab specific libraries not found. Skipping.")

In [4]:
drive.mount('/content/drive',force_remount = True)

Mounted at /content/drive


# Model Factory

In [5]:
class BaseEmbeddingModel:
    """Standard Interface for all models"""
    def __init__(self, device=None):
        self.device = device if device else ("cuda" if torch.cuda.is_available() else "cpu")
        self.device = torch.device(self.device)

    def get_transforms(self):
        """Returns the torchvision-compatible transform for the dataset"""
        raise NotImplementedError

    def embed_image(self, image_tensor):
        """Accepts (B, C, H, W) tensor, returns (B, D) normalized embedding"""
        raise NotImplementedError

    def embed_text(self, text_list):
        """Optional: Accepts list of strings, returns (B, D) normalized embedding"""
        raise NotImplementedError("This model does not support text embeddings.")

# ==========================================
# 1. ResNet50 (The Baseline)
# ==========================================
class ResNetEmbedder(BaseEmbeddingModel):
    def __init__(self, device=None):
        super().__init__(device)
        print(f"Loading ResNet50 on {self.device}...")
        weights = models.ResNet50_Weights.DEFAULT
        backbone = models.resnet50(weights=weights)
        # Remove the classification head (fc)
        self.model = nn.Sequential(*(list(backbone.children())[:-1]))
        self.model.to(self.device)
        self.model.eval()
        self.transforms = weights.transforms()

    def get_transforms(self):
        return self.transforms

    def embed_image(self, image_tensor):
        with torch.no_grad():
            # Output: (B, 2048, 1, 1) -> Flatten to (B, 2048)
            features = self.model(image_tensor.to(self.device))
            features = features.flatten(1)
            return torch.nn.functional.normalize(features, p=2, dim=1)

# ==========================================
# 2. ConvNeXt (Modern CNN)
# ==========================================
class ConvNeXtEmbedder(BaseEmbeddingModel):
    def __init__(self, version='base', device=None):
        super().__init__(device)
        print(f"Loading ConvNeXt-{version} on {self.device}...")

        # Select correct weights
        if version == 'base':
            weights = models.ConvNeXt_Base_Weights.DEFAULT
            self.model = models.convnext_base(weights=weights)
        elif version == 'tiny':
            weights = models.ConvNeXt_Tiny_Weights.DEFAULT
            self.model = models.convnext_tiny(weights=weights)
        else:
            raise ValueError("Supported versions: 'base', 'tiny'")

        # Replace classifier with Identity to get features
        self.model.classifier = nn.Identity()
        self.model.to(self.device)
        self.model.eval()
        self.transforms = weights.transforms()

    def get_transforms(self):
        return self.transforms

    def embed_image(self, image_tensor):
        with torch.no_grad():
            features = self.model(image_tensor.to(self.device))
            # Safety: Ensure output is (B, D) by flattening any trailing dims
            features = features.flatten(1)
            return torch.nn.functional.normalize(features, p=2, dim=1)

# ==========================================
# 3. DINOv2 (Visual King)
# ==========================================
class DINOv2Embedder(BaseEmbeddingModel):
    def __init__(self, size='base', device=None):
        super().__init__(device)

        # Robust Model ID Mapping
        model_map = {
            'small': 'facebook/dinov2-small',
            'base':  'facebook/dinov2-base',
            'large': 'facebook/dinov2-large',
            'giant': 'facebook/dinov2-giant'
        }

        if size not in model_map:
            raise ValueError(f"Unknown DINOv2 size: {size}. Choose from {list(model_map.keys())}")

        model_id = model_map[size]
        print(f"Loading DINOv2 ({model_id}) on {self.device}...")

        self.processor = AutoImageProcessor.from_pretrained(model_id)
        self.model = AutoModel.from_pretrained(model_id).to(self.device)
        self.model.eval()

    def get_transforms(self):
        # Wrap HF processor to work like a standard Transform
        def hf_transform(img):
            # Processor returns dict with 'pixel_values', we need just the tensor
            # Note: This runs on CPU per item (slower but simple)
            return self.processor(images=img, return_tensors="pt")['pixel_values'].squeeze(0)
        return hf_transform

    def embed_image(self, image_tensor):
        with torch.no_grad():
            outputs = self.model(pixel_values=image_tensor.to(self.device))
            # DINOv2 uses the [CLS] token (index 0)
            features = outputs.last_hidden_state[:, 0, :]
            return torch.nn.functional.normalize(features, p=2, dim=1)

# ==========================================
# 4. SigLIP (Concept King)
# ==========================================
class SigLIPEmbedder(BaseEmbeddingModel):
    def __init__(self, model_id="google/siglip-base-patch16-224", device=None):
        super().__init__(device)
        print(f"Loading SigLIP ({model_id}) on {self.device}...")
        self.model = SiglipModel.from_pretrained(model_id).to(self.device)
        self.processor = SiglipProcessor.from_pretrained(model_id)
        self.model.eval()

    def get_transforms(self):
        def hf_transform(img):
            return self.processor(images=img, return_tensors="pt")['pixel_values'].squeeze(0)
        return hf_transform

    def embed_image(self, image_tensor):
        with torch.no_grad():
            features = self.model.get_image_features(pixel_values=image_tensor.to(self.device))
            return torch.nn.functional.normalize(features, p=2, dim=1)

    def embed_text(self, text_list):
        with torch.no_grad():
            # Use padding=True instead of max_length for efficiency
            inputs = self.processor(text=text_list, return_tensors="pt", padding=True, truncation=True)
            inputs = {k: v.to(self.device) for k, v in inputs.items()}
            features = self.model.get_text_features(**inputs)
            return torch.nn.functional.normalize(features, p=2, dim=1)

# ==========================================
# 5. FashionCLIP (Domain Specialist)
# ==========================================
class FashionCLIPEmbedder(BaseEmbeddingModel):
    def __init__(self, device=None):
        super().__init__(device)
        print(f"Loading FashionCLIP on {self.device}...")
        self.model = CLIPModel.from_pretrained("patrickjohncyh/fashion-clip").to(self.device)
        self.processor = CLIPProcessor.from_pretrained("patrickjohncyh/fashion-clip")
        self.model.eval()

    def get_transforms(self):
        def hf_transform(img):
            return self.processor(images=img, return_tensors="pt")['pixel_values'].squeeze(0)
        return hf_transform

    def embed_image(self, image_tensor):
        with torch.no_grad():
            features = self.model.get_image_features(pixel_values=image_tensor.to(self.device))
            return torch.nn.functional.normalize(features, p=2, dim=1)

    def embed_text(self, text_list):
        with torch.no_grad():
            inputs = self.processor(text=text_list, return_tensors="pt", padding=True, truncation=True)
            inputs = {k: v.to(self.device) for k, v in inputs.items()}
            features = self.model.get_text_features(**inputs)
            return torch.nn.functional.normalize(features, p=2, dim=1)

# ==========================================
# FACTORY CLASS
# ==========================================
class ModelFactory:
    @staticmethod
    def get_model(name, device=None):
        name = name.lower()
        if name == 'resnet50':
            return ResNetEmbedder(device=device)
        elif name == 'convnext':
            return ConvNeXtEmbedder(version='base', device=device)
        elif name == 'dinov2':
            return DINOv2Embedder(size='base', device=device)
        elif name == 'siglip':
            return SigLIPEmbedder(device=device)
        elif name == 'fashionclip':
            return FashionCLIPEmbedder(device=device)
        else:
            raise ValueError(f"Unknown model: {name}. Options: [resnet50, convnext, dinov2, siglip, fashionclip]")

# Feature Extraction

In [6]:
# ==========================================
# 1. CONFIGURATION (Dynamic globals removed)
# ==========================================
BASE_DIR        = "/content/drive/MyDrive/Product_Recommender_End_to_End_with_virtual_dressing"
MANIFEST_DL_CSV = f"{BASE_DIR}/Data/Raw/Images/images_manifest.csv.gz"
HF_REPO_ID      = "PirateKing0402/Amazon-fashion-image-tars"

# Runtime
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
NUM_WORKERS = min(8, max(2, (os.cpu_count() or 8)//2))
# NUM_WORKERS = 2
BATCH_SIZE  = 64
ROWS_PER_WRITE = 20_000
PARQUET_COMPRESSION = "snappy"
TAR_BATCH_SIZE = 8

print("Device:", DEVICE, "| num_workers:", NUM_WORKERS)

Device: cpu | num_workers: 4


In [7]:
# ==========================================
# 2. HELPER FUNCTIONS (Untouched logic)
# ==========================================
def list_shards(_root_ignored: str) -> List[str]:
    """Return shard folder names under tars/ in the HF repo (keeps original signature)."""
    files = list_repo_files(HF_REPO_ID, repo_type="dataset")
    shards = set()
    for f in files:
        if f.startswith("tars/") and f.endswith(".tar"):
            parts = f.split("/")
            if len(parts) >= 3:
                shards.add(parts[1])
    return sorted(shards)

def hf_tar_url_batches(repo_id: str, shard_name: str, batch_size: int = TAR_BATCH_SIZE, root_prefix: str = "tars/"):
    """Yield small lists of HTTPS TAR URLs for tars/<shard_name>/*.tar (caps RAM)."""
    files = list_repo_files(repo_id, repo_type="dataset")
    tars = [f for f in files if f.startswith(f"{root_prefix}{shard_name}/") and f.endswith(".tar")]
    tars.sort()
    for i in range(0, len(tars), batch_size):
        batch = tars[i:i+batch_size]
        yield [hf_hub_url(repo_id, f, repo_type="dataset") for f in batch]

def make_wds_for_shard_batch(tar_urls: List[str]):
    """
    Build a WebDataset over a small list of TAR URLs (HF).
    Yields (key:str, asin:str, PIL.Image).
    """
    # -------------------------------------------------------------------------
    # RESILIENCE FIX:
    # 1. empty_check=False: Ignore empty/corrupt files
    # 2. handler=wds.warn_and_continue: If connection drops, log it and move to next TAR
    # -------------------------------------------------------------------------
    ds = (wds.WebDataset(
            tar_urls,
            shardshuffle=False,
            empty_check=False,
            handler=wds.warn_and_continue
          )
          .select(lambda s: (("jpg" in s) or ("png" in s)))
          .decode("pil", handler=wds.warn_and_continue))

    def mapper(sample: Dict[str, Any]) -> Tuple[str, str, Image.Image]:
        key  = sample["__key__"]
        base = key.split("/")[-1]
        asin = base.split("_")[0]
        img  = (sample["jpg"] if "jpg" in sample else sample["png"]).convert("RGB")
        return key, asin, img

    return ds.map(mapper)

# MODIFIED: Now accepts transform_fn to support different models
def collate_batch_dynamic(samples, transform_fn=None):
    """
    Applies transform_fn to images immediately (Worker side).
    Returns stacked Tensor, saving RAM in the queue.
    """
    keys = [s[0] for s in samples]
    ids  = [s[1] for s in samples]
    # Apply transform immediately and stack into a Batch Tensor
    imgs = torch.stack([transform_fn(s[2]) for s in samples])
    return keys, ids, imgs

def np2fixed_list_2d(x2d: np.ndarray, dim: int) -> pa.FixedSizeListArray:
    # MODIFIED: Removed global assert, accepts dim argument
    flat = pa.array(x2d.reshape(-1), type=pa.float32())
    return pa.FixedSizeListArray.from_arrays(flat, dim)

def expected_images_for_shard_from_manifest(manifest_gz: str, shard_name: str) -> int:
    total = 0
    with gzip.open(manifest_gz, "rt", newline="") as fh:
        r = csv.DictReader(fh)
        for row in r:
            if row.get("ok") == "1" and row.get("shard") == shard_name:
                total += 1
    return total

In [8]:
# ==========================================
# 3. DB LOGIC (Untouched except path injection)
# ==========================================
def _db_paths(root_dir: str, shard_name: str):
    # MODIFIED: Accepts root_dir instead of global SHARDS_STORE
    out_dir = os.path.join(root_dir, shard_name)
    emb_dir = os.path.join(out_dir, "emb_parts")
    man_dir = os.path.join(out_dir, "emb_manifest")
    Path(emb_dir).mkdir(parents=True, exist_ok=True)
    Path(man_dir).mkdir(parents=True, exist_ok=True)
    return out_dir, emb_dir, man_dir, os.path.join(out_dir, "processed.sqlite")

def open_db(db_path: str):
    # MODIFIED: Accepts full db_path directly
    conn = sqlite3.connect(db_path, isolation_level=None, check_same_thread=False)
    cur  = conn.cursor()
    cur.execute("PRAGMA journal_mode=WAL")
    cur.execute("PRAGMA synchronous=OFF")  # <- as requested (fastest)
    cur.execute("""
        CREATE TABLE IF NOT EXISTS processed (
            __key__ TEXT PRIMARY KEY,
            status  TEXT CHECK(status IN ('pending','ok')) NOT NULL
        )
    """)
    return conn

def cleanup_stale_pending(conn):
    cur = conn.cursor()
    cur.execute("DELETE FROM processed WHERE status='pending'")

def count_ok(conn) -> int:
    cur = conn.cursor()
    cur.execute("SELECT COUNT(*) FROM processed WHERE status='ok'")
    return int(cur.fetchone()[0])

def seed_ok_from_existing_embeddings(conn, emb_dir: str):
    """
    One quick pass to ensure DB matches any already-written embeddings.
    """
    # MODIFIED: Accepts emb_dir directly
    part_glob = os.path.join(emb_dir, "part-*.parquet")
    parts = sorted(glob.glob(part_glob))
    if not parts:
        return

    cur = conn.cursor()
    for fp in parts:
        try:
            pf = pq.ParquetFile(fp)
            cols = pf.schema_arrow.names
            key_col = "__key__" if "__key__" in cols else None
            if key_col is None:
                continue
            for rg in range(pf.num_row_groups):
                tbl = pf.read_row_group(rg, columns=[key_col])
                keys = [(str(k),) for k in tbl[key_col].to_pylist()]
                cur.executemany(
                    "INSERT OR IGNORE INTO processed(__key__, status) VALUES (?, 'ok')", keys
                )
        except Exception:
            pass

def reserve_pending(conn, keys, chunk_size=512):
    """
    Try to reserve each key (INSERT OR IGNORE) in small chunks.
    Return indices in `keys` that were actually inserted now (i.e., new work).
    """
    if not keys:
        return []

    keep_idx = []
    cur = conn.cursor()

    for start in range(0, len(keys), chunk_size):
        chunk = keys[start:start + chunk_size]
        for off, k in enumerate(chunk):
            cur.execute(
                "INSERT OR IGNORE INTO processed(__key__, status) VALUES (?, 'pending')",
                (k,)
            )
            if cur.rowcount == 1:
                keep_idx.append(start + off)

    return keep_idx

def mark_ok(conn, keys: List[str]):
    if not keys:
        return
    cur = conn.cursor()
    CHUNK = 500
    for i in range(0, len(keys), CHUNK):
        chunk = keys[i:i+CHUNK]
        q = ",".join(["?"]*len(chunk))
        cur.execute(f"UPDATE processed SET status='ok' WHERE __key__ IN ({q}) AND status='pending'", chunk)

def release_pending(conn, keys: List[str]):
    if not keys:
        return
    cur = conn.cursor()
    CHUNK = 500
    for i in range(0, len(keys), CHUNK):
        chunk = keys[i:i+CHUNK]
        q = ",".join(["?"]*len(chunk))
        cur.execute(f"DELETE FROM processed WHERE __key__ IN ({q}) AND status='pending'", chunk)

In [23]:
# 1. MANIFEST CHECKER CLASS (RAM Optimized)
class ManifestChecker:
    def __init__(self, manifest_path, shard_name):
        print(f"⚙️ Initializing ManifestChecker (RAM) for shard: {shard_name}...")
        self.con = duckdb.connect(database=':memory:')

        # Using rf""" for regex backslashes
        query = rf"""
            CREATE TABLE manifest_index AS
            SELECT
                split_part(tar_path, '/', -1) as tar_filename,
                regexp_replace(tar_member, '\.[^/]+$', '') as clean_key
            FROM read_csv('{manifest_path}', AUTO_DETECT=TRUE)
            WHERE shard = '{shard_name}'
        """
        try:
            self.con.execute(query)
            self.con.execute("CREATE INDEX idx_tar ON manifest_index(tar_filename)")
            count = self.con.execute("SELECT count(*) FROM manifest_index").fetchone()[0]
            print(f"   ✅ Indexed {count} images in RAM.")
        except Exception as e:
            print(f"❌ Error initializing DuckDB: {e}")
            raise e

    def get_missing_keys_for_url(self, hf_url, conn):
        decoded_url = urllib.parse.unquote(hf_url)
        tar_filename = os.path.basename(decoded_url)

        # Query DuckDB
        rows = self.con.execute(
            "SELECT clean_key FROM manifest_index WHERE tar_filename = ?",
            [tar_filename]
        ).fetchall()
        expected_keys = [r[0] for r in rows]

        if not expected_keys: return set()

        # Query SQLite (Chunked)
        found_keys = set()
        chunk_size = 900
        cursor = conn.cursor()
        for i in range(0, len(expected_keys), chunk_size):
            chunk = expected_keys[i:i+chunk_size]
            placeholders = ','.join(['?'] * len(chunk))
            query = f"SELECT __key__ FROM processed WHERE __key__ IN ({placeholders}) AND status='ok'"
            cursor.execute(query, chunk)
            found_keys.update({row[0] for row in cursor.fetchall()})
        return set(expected_keys) - found_keys

    def close(self):
        self.con.close()
        print("   🗑️ ManifestChecker RAM released.")

In [24]:
# !mkdir -p /tmp/tarcheck
# %cd /tmp/tarcheck

# URL="https://huggingface.co/datasets/PirateKing0402/Amazon-fashion-image-tars/resolve/main/tars/boys/boys-00008.tar"

# # 1) download with retries (fail if incomplete)
# !curl -L --fail --retry 5 --retry-delay 2 --connect-timeout 30 -o boys-00008.tar "$URL"

# # 2) tar listing (touches entire archive structure)
# !tar -tf boys-00008.tar > /tmp/tar_members.txt

# # 3) quick stats
# !wc -l /tmp/tar_members.txt

# # 4) Final check
# !tar -tf boys-00008.tar

In [25]:
# import time
# import torch
# from tqdm.auto import tqdm

# def benchmark_raw_throughput(shard_name, model_name, limit_tars=5, apply_transform=False):
#     print(f"🧪 BENCHMARKING: Shard='{shard_name}' | Transform={apply_transform}")

#     # 1. Get a fresh batch of TAR URLs
#     tar_batch_gen = hf_tar_url_batches(HF_REPO_ID, shard_name, batch_size=limit_tars)
#     tar_urls = next(tar_batch_gen)
#     print(f"   📂 Testing on {len(tar_urls)} TAR files.")

#     # 2. Setup Transform (if requested)
#     transform = None
#     if apply_transform:
#         # Get the actual transform from your wrapper
#         transform = wrapper_instance.get_transforms()
#         print("   ⚙️ Transformations ENABLED (CPU intensive)")
#     else:
#         print("   🚀 Transformations DISABLED (Pure Network + Decode)")

#     # 3. Create Dataset (No DataLoader, just raw Iterator)
#     ds = make_wds_for_shard_batch(tar_urls)

#     # 4. Run Loop
#     count = 0
#     start_time = time.time()
#     window_start = time.time()
#     window_size = 50  # Measure speed every 50 images

#     print("\n   👇 Starting Stream...")
#     try:
#         for key, asin, img in ds:
#             # --- MEASUREMENT POINT A: Fetch & Decode Done ---

#             # Optional: Simulate Transform Cost
#             if transform:
#                 _ = transform(img)

#             count += 1

#             # Periodic Reporting
#             if count % window_size == 0:
#                 now = time.time()
#                 window_time = now - window_start
#                 avg_speed = window_size / window_time

#                 total_elapsed = now - start_time
#                 total_avg = count / total_elapsed

#                 print(f"   📸 Img {count}: Current Speed = {avg_speed:.1f} img/s | Overall = {total_avg:.1f} img/s")

#                 window_start = now

#     except KeyboardInterrupt:
#         print("\n   🛑 Stopped by User.")
#     except Exception as e:
#         print(f"\n   ❌ Error: {e}")

#     total_time = time.time() - start_time
#     print(f"\n✅ DONE. Processed {count} images in {total_time:.1f}s.")
#     if total_time > 0:
#         print(f"📊 Final Average Speed: {count / total_time:.2f} img/s")

# # ==========================================
# # RUN THE TEST
# # ==========================================

# # TEST 1: Raw Network + Decoding (Is Hugging Face slow?)
# benchmark_raw_throughput("boys", "convnext", limit_tars=4, apply_transform=False)

In [26]:
# 3. MAIN FUNCTION
@torch.inference_mode()
def embed_shard_to_parquet(shard_name: str,
                           model_name: str,
                           wrapper_instance,
                           batch_size: int = BATCH_SIZE,
                           num_workers: int = NUM_WORKERS,
                           rows_per_write: int = ROWS_PER_WRITE,
                           compression: str = PARQUET_COMPRESSION,
                           tar_batch_size: int = TAR_BATCH_SIZE,
                           debug_mode: bool = False,
                           check_gpu_util: bool = False,
                           gpu_check_every_images: int = 500):

    # Setup Paths
    model_shards_root = f"{BASE_DIR}/Features/{model_name}/shards"
    out_dir, emb_dir, man_dir, db_path = _db_paths(model_shards_root, shard_name)

    # Setup DB
    conn = open_db(db_path)
    cleanup_stale_pending(conn)
    seed_ok_from_existing_embeddings(conn, emb_dir)
    ok_so_far = count_ok(conn)

    # --- INIT MANIFEST CHECKER ---
    manifest_checker = ManifestChecker(MANIFEST_DL_CSV, shard_name)

    # Setup Progress
    try:
        # Count only UNIQUE keys
        exp_total = manifest_checker.con.execute("SELECT count(DISTINCT clean_key) FROM manifest_index").fetchone()[0]
    except: exp_total = None
    remaining = None if exp_total is None else max(0, exp_total - ok_so_far)

    print(f"[{model_name}::{shard_name}] DONE: {ok_so_far} | REMAINING: {remaining}")
    pbar = tqdm(total=remaining, desc=f"{model_name} -> {shard_name}", unit="img")

    # Determine Dim
    dummy = torch.zeros(1, 3, 224, 224).to(DEVICE)
    try:
        with torch.no_grad():
            EMB_DIM = wrapper_instance.embed_image(dummy).shape[1]
    except: EMB_DIM = 2048

    # Schemas
    EMB_SCHEMA = pa.schema([
        pa.field("row_idx", pa.int64()),
        pa.field("__key__", pa.string()),
        pa.field("asin",    pa.string()),
        pa.field("shard",   pa.string()),
        pa.field("emb",     pa.list_(pa.float32(), EMB_DIM)),
    ])

    MANIFEST_SCHEMA = pa.schema([
        pa.field("row_idx", pa.int64()),
        pa.field("__key__", pa.string()),
        pa.field("asin",    pa.string()),
        pa.field("shard",   pa.string()),
        pa.field("ok",      pa.int8()),
        pa.field("error",   pa.string()),
        pa.field("part",    pa.string()),
        pa.field("ts_ms",   pa.int64()),
    ])

    # Buffers
    run_id   = f"{time.strftime('%Y%m%d-%H%M%S')}-{np.random.randint(1000,9999)}"
    part_seq = 0
    next_row = ok_so_far
    buf_keys, buf_asins, buf_embs, buf_rows = [], [], [], 0

    def flush_buffers():
        nonlocal buf_keys, buf_asins, buf_embs, buf_rows, part_seq, next_row
        if buf_rows == 0: return
        embs = np.vstack(buf_embs).astype(np.float32, copy=False)
        arr_row   = pa.array(np.arange(next_row, next_row + buf_rows, dtype=np.int64))
        arr_key   = pa.array(buf_keys,  type=pa.string())
        arr_asin  = pa.array(buf_asins, type=pa.string())
        arr_shard = pa.array([shard_name]*buf_rows, type=pa.string())
        arr_emb   = np2fixed_list_2d(embs, EMB_DIM)
        part_path = os.path.join(emb_dir, f"part-{run_id}-{part_seq:05d}.parquet")
        emb_tbl = pa.Table.from_arrays([arr_row, arr_key, arr_asin, arr_shard, arr_emb], schema=EMB_SCHEMA)
        pq.write_table(emb_tbl, part_path, compression=compression)

        ts = int(time.time()*1000)
        man_tbl = pa.Table.from_arrays([
            arr_row, arr_key, arr_asin, arr_shard,
            pa.array([1]*buf_rows, type=pa.int8()),
            pa.array([""]*buf_rows, type=pa.string()),
            pa.array([part_path]*buf_rows, type=pa.string()),
            pa.array([ts]*buf_rows, type=pa.int64())
        ], schema=MANIFEST_SCHEMA)
        pq.write_table(man_tbl, os.path.join(man_dir, f"manifest-{run_id}-{part_seq:05d}.parquet"))

        next_row += buf_rows
        part_seq += 1
        buf_keys.clear(); buf_asins.clear(); buf_embs.clear(); buf_rows = 0

    seen_written = 0
    t0 = time.time()

    # ===================== GPU Helper =====================
    last_gpu_check_seen = 0
    last_gpu_check_t = t0
    def _gpu_smi_snapshot():
        if DEVICE != "cuda": return "GPU=N/A"
        try: torch.cuda.synchronize()
        except: pass
        try:
            out = subprocess.check_output(
                ["nvidia-smi", "--query-gpu=utilization.gpu,memory.used,memory.total", "--format=csv,noheader,nounits"],
                text=True).strip()
            return f"GPU={out}"
        except: return "GPU=Unknown"
    # =======================================================

    # 4. PREPARE TRANSFORM
    specific_transform = wrapper_instance.get_transforms()
    collate_fn = functools.partial(collate_batch_dynamic, transform_fn=specific_transform)

    try:
        batch_counter = 0
        for tar_urls in hf_tar_url_batches(HF_REPO_ID, shard_name, batch_size=tar_batch_size):
            batch_counter += 1

            active_tar_urls = []

            # --- SMART FILTERING (Manifest + DB) ---
            for tar_url in tar_urls:
                # We check if TAR has any work left.
                # We do NOT build a registry for per-key filtering anymore (Reserve Pending handles that).
                missing = manifest_checker.get_missing_keys_for_url(tar_url, conn)
                if len(missing) > 0:
                    active_tar_urls.append(tar_url)

            if not active_tar_urls:
                if debug_mode:
                    print(f"⏭️ Skipping batch {batch_counter} (Verified Done)...", end='\r')
                continue

            if debug_mode:
                print(f"🔍 Batch {batch_counter}: {len(active_tar_urls)} TARs active.")

            # --- USE ORIGINAL HELPER (With warn_and_continue) ---
            ds = make_wds_for_shard_batch(active_tar_urls)

            dl_kwargs = dict(batch_size=batch_size, num_workers=num_workers,
                                pin_memory=True, collate_fn=collate_fn)
            if num_workers > 0:
                dl_kwargs.update(persistent_workers=True, prefetch_factor=4)

            dl = torch.utils.data.DataLoader(ds, **dl_kwargs)

            for keys, asins, imgs_tensor in dl:

                # 1. LOCKING (DB Side)
                # This automatically filters out keys that are already 'ok' or 'pending'
                hold_idx = reserve_pending(conn, keys)
                if not hold_idx:
                    continue

                # 2. SLICING (Critical Fix)
                # Only process what we successfully locked
                if len(hold_idx) != len(keys):
                    keys = [keys[i] for i in hold_idx]
                    asins = [asins[i] for i in hold_idx]
                    imgs_tensor = imgs_tensor[hold_idx]

                # 3. INFERENCE
                embs = wrapper_instance.embed_image(imgs_tensor).detach().cpu().numpy().astype(np.float32)

                # 4. BUFFER
                B = embs.shape[0]
                buf_keys.extend(keys)
                buf_asins.extend(asins)
                buf_embs.append(embs)
                buf_rows += B
                seen_written += B
                pbar.update(B)

                # GPU Check
                if check_gpu_util and (seen_written - last_gpu_check_seen) >= gpu_check_every_images:
                    now = time.time()
                    interval_s = max(now - last_gpu_check_t, 1e-9)
                    interval_imgs = seen_written - last_gpu_check_seen
                    interval_ips = interval_imgs / interval_s
                    smi = _gpu_smi_snapshot()
                    print(f"\n🧪 SANITY[{model_name}::{shard_name}] imgs={seen_written} | "
                            f"+{interval_imgs} imgs in {interval_s:.1f}s => {interval_ips:.2f} img/s | {smi}\n")
                    last_gpu_check_seen = seen_written
                    last_gpu_check_t = now

                if buf_rows >= rows_per_write:
                    keys_snap = list(buf_keys)
                    flush_buffers()
                    mark_ok(conn, keys_snap)

            del dl, ds
            if DEVICE == "cuda":
                torch.cuda.empty_cache()

    except (Exception, KeyboardInterrupt) as e:
        release_pending(conn, buf_keys)
        print("Cleaned up pending keys.")
        if isinstance(e, KeyboardInterrupt):
            raise
        print(f"ERROR: {e}")

    finally:
        # CLEANUP
        manifest_checker.close()

        if buf_rows > 0:
            keys_snap = list(buf_keys)
            flush_buffers()
            mark_ok(conn, keys_snap)
        pbar.close()
        conn.close()

    elapsed = time.time() - t0
    print(f"\n[{shard_name}] DONE. +{seen_written} images in {elapsed:.1f}s")

In [27]:
# ==========================================
# 7. EXECUTION ENTRY POINT
# ==========================================
def run_embedding_pipeline(models_to_run: List[str], shards_limit: int = None,debug_mode = False,check_gpu_util = False,gpu_check_every_images = 500):
    """
    Main entry point to run multiple models sequentially.
    """
    all_shards = list_shards(HF_REPO_ID)
    if shards_limit:
        all_shards = all_shards[:shards_limit]

    print(f"🎯 Found {len(all_shards)} shards to process.")

    for model_name in models_to_run:
        print(f"\n{'='*40}")
        print(f"🚀 STARTING MODEL: {model_name.upper()}")
        print(f"{'='*40}")

        try:
            wrapper = ModelFactory.get_model(model_name, device=DEVICE)
        except Exception as e:
            print(f"⚠️ Failed to load {model_name}: {e}")
            continue

        for shard in all_shards:
            embed_shard_to_parquet(
                shard_name=shard,
                model_name=model_name,
                wrapper_instance=wrapper,
                batch_size=BATCH_SIZE,
                debug_mode = debug_mode,
                check_gpu_util=check_gpu_util,
                gpu_check_every_images=gpu_check_every_images,
            )

        del wrapper
        gc.collect()
        torch.cuda.empty_cache()

In [28]:
run_embedding_pipeline(models_to_run=["convnext"],debug_mode=True,check_gpu_util=True, gpu_check_every_images=500)

🎯 Found 11 shards to process.

🚀 STARTING MODEL: CONVNEXT
Loading ConvNeXt-base on cpu...
⚙️ Initializing ManifestChecker (RAM) for shard: baby...
   ✅ Indexed 46015 images in RAM.
[convnext::baby] DONE: 46015 | REMAINING: 0


convnext -> baby: 0img [00:00, ?img/s]

   🗑️ ManifestChecker RAM released.

[baby] DONE. +0 images in 0.3s
⚙️ Initializing ManifestChecker (RAM) for shard: boys...
   ✅ Indexed 70068 images in RAM.
[convnext::boys] DONE: 70068 | REMAINING: 0


convnext -> boys: 0img [00:00, ?img/s]

   🗑️ ManifestChecker RAM released.

[boys] DONE. +0 images in 0.5s
⚙️ Initializing ManifestChecker (RAM) for shard: clothing|mens...
   ✅ Indexed 288107 images in RAM.
[convnext::clothing|mens] DONE: 288057 | REMAINING: 0


...

convnext -> jewelry|womens:   0%|          | 0/16425 [00:00<?, ?img/s]

🔍 Batch 1: 3 TARs active.


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


🔍 Batch 2: 4 TARs active.

🧪 SANITY[convnext::jewelry|womens] imgs=512 | +512 imgs in 503.9s => 1.02 img/s | GPU=N/A


🧪 SANITY[convnext::jewelry|womens] imgs=1024 | +512 imgs in 98.3s => 5.21 img/s | GPU=N/A


🧪 SANITY[convnext::jewelry|womens] imgs=1536 | +512 imgs in 99.4s => 5.15 img/s | GPU=N/A


🧪 SANITY[convnext::jewelry|womens] imgs=2048 | +512 imgs in 99.5s => 5.15 img/s | GPU=N/A

🔍 Batch 3: 2 TARs active.

🧪 SANITY[convnext::jewelry|womens] imgs=2576 | +528 imgs in 124.6s => 4.24 img/s | GPU=N/A


🧪 SANITY[convnext::jewelry|womens] imgs=3088 | +512 imgs in 102.0s => 5.02 img/s | GPU=N/A


🧪 SANITY[convnext::jewelry|womens] imgs=3600 | +512 imgs in 103.4s => 4.95 img/s | GPU=N/A


🧪 SANITY[convnext::jewelry|womens] imgs=4112 | +512 imgs in 105.7s => 4.85 img/s | GPU=N/A


🧪 SANITY[convnext::jewelry|womens] imgs=4624 | +512 imgs in 103.0s => 4.97 img/s | GPU=N/A


🧪 SANITY[convnext::jewelry|womens] imgs=5136 | +512 imgs in 103.3s => 4.96 img/s | GPU=N/A


🧪 SANITY[convnext::je

DatabaseError: database disk image is malformed

In [ ]:
from google.colab import runtime
print("😴 Job done. Going to sleep now...")
runtime.unassign()

## Checking Manifest File of the embeddings generated.

In [ ]:
MODEL_NAME = "ConvNext"

BASE_DIR        = "/content/drive/MyDrive/Product_Recommender_End_to_End_with_virtual_dressing"
SHARDS_STORE    = f"{BASE_DIR}/Features/{MODEL_NAME.lower()}/shards"
DOWN_MANIFEST   = f"{BASE_DIR}/Data/Raw/Images/images_manifest.csv.gz"   # original download manifest with 'ok' and 'shard'
HF_REPO_ID = "PirateKing0402/Amazon-fashion-image-tars"

In [ ]:
def list_shards(_root_ignored: str) -> List[str]:
    """Return shard folder names under tars/ in the HF repo (keeps original signature)."""
    files = list_repo_files(HF_REPO_ID, repo_type="dataset")
    shards = set()
    for f in files:
        # Expect paths like: tars/<shard>/<file>.tar
        if f.startswith("tars/") and f.endswith(".tar"):
            parts = f.split("/")
            if len(parts) >= 3:
                shards.add(parts[1])
    return sorted(shards)

In [ ]:
shards = list_shards("/ignored")
print("HF shards:", shards[:10], "… total:", len(shards))
SHARD = shards[0] if shards else None
SHARD

HF shards: ['baby', 'boys', 'clothing|mens', 'clothing|womens', 'girls', 'jewelry|mens', 'jewelry|womens', 'shoes|mens', 'shoes|womens', 'unisex-adult'] … total: 11


'baby'

In [ ]:
okks = []
for SHARD in shards:

    man_dir = os.path.join(SHARDS_STORE, SHARD, "emb_manifest")
    emb_dir = os.path.join(SHARDS_STORE, SHARD, "emb_parts")

    print("man_dir exists:", os.path.isdir(man_dir), "| emb_dir exists:", os.path.isdir(emb_dir))

    part_paths = sorted(glob.glob(os.path.join(man_dir, "manifest-*.parquet")))
    print(f"{SHARD}: {len(part_paths)} manifest part files")

    ok = 0
    err = 0
    err_msgs = Counter()
    keys_seen = set()
    dups = 0

    for fp in tqdm(part_paths, desc="Scan manifests"):
        pf = pq.ParquetFile(fp)
        for rg in range(pf.num_row_groups):
            tbl = pf.read_row_group(rg, columns=["__key__","ok","error"])
            k = tbl["__key__"].to_pylist()
            o = tbl["ok"].to_numpy()
            e = tbl["error"].to_pylist()
            for i, kk in enumerate(k):
                if kk in keys_seen:
                    dups += 1
                else:
                    keys_seen.add(kk)
                if o[i] == 1:
                    ok += 1
                else:
                    err += 1
                    err_msgs[e[i]] += 1
    okks.append(ok)

    print(f"\n[{SHARD}] ok={ok:,}  errors={err:,}  duplicate_keys_in_manifest={dups}")
    print("Top error reasons:", err_msgs.most_common(10))


man_dir exists: True | emb_dir exists: True
baby: 6 manifest part files


Scan manifests:   0%|          | 0/6 [00:00<?, ?it/s]


[baby] ok=45,757  errors=0  duplicate_keys_in_manifest=0
Top error reasons: []
man_dir exists: False | emb_dir exists: False
boys: 0 manifest part files


Scan manifests: 0it [00:00, ?it/s]


[boys] ok=0  errors=0  duplicate_keys_in_manifest=0
Top error reasons: []
man_dir exists: False | emb_dir exists: False
clothing|mens: 0 manifest part files


...


[shoes|womens] ok=0  errors=0  duplicate_keys_in_manifest=0
Top error reasons: []
man_dir exists: False | emb_dir exists: False
unisex-adult: 0 manifest part files


Scan manifests: 0it [00:00, ?it/s]


[unisex-adult] ok=0  errors=0  duplicate_keys_in_manifest=0
Top error reasons: []
man_dir exists: False | emb_dir exists: False
unisex-child: 0 manifest part files


Scan manifests: 0it [00:00, ?it/s]


[unisex-child] ok=0  errors=0  duplicate_keys_in_manifest=0
Top error reasons: []


In [ ]:
for i,SHARD in enumerate(shards):
    expected = 0
    with gzip.open(DOWN_MANIFEST, "rt", newline="") as fh:
        r = csv.DictReader(fh)
        for row in r:
            if row.get("ok") == "1" and row.get("shard") == SHARD:
                expected += 1

    print(f"[{SHARD}] expected from download manifest: {expected:,}")
    if expected:
        print(f"coverage: {okks[i]/expected:.2%}")


[baby] expected from download manifest: 46,015
coverage: 99.44%
[boys] expected from download manifest: 70,068
coverage: 0.00%


KeyboardInterrupt: 

# Finding Recommendations.